In [1]:
from datetime import datetime
import sqlalchemy as sa
import pyodbc
import urllib
import logging

In [2]:
#adding a file for basic logging - we want to se possible errors from the DB side

logging.basicConfig(
    filename='weather_etl.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s', #timestamp, level (INFO etc.), message
    datefmt='%Y-%m-%d %H:%M:%S'
)

In [3]:
#in this part we are connecting to the database and executing procedures to load datamart (where needed)

params = urllib.parse.quote_plus("DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost\\SQLEXPRESS;DATABASE=WeatherDB;Trusted_Connection=yes;") #translating special characters into standardized URL codes

try:
    engine = sa.create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
    print("connected to WeatherDB")
except Exception as e:
    print(f"error occurred: {e}")

try:
    with engine.begin() as conn: #important so that changes are commited
    #checking date dimension because the way it's populated doesn't depend on the data
        date_count = conn.execute(sa.text("SELECT COUNT(*) FROM dbo.DIM_DATE")).scalar()
        if date_count == 0:
            print("executing dbo.PROC_POPULATE_DIM_DATE")
            conn.execute(sa.text("EXEC dbo.PROC_POPULATE_DIM_DATE;"))
        else:
            print("DIM_DATE already has data")
        
    #checking unit dimension because the way it's populated doesn't depend on the data
        unit_count = conn.execute(sa.text("SELECT COUNT(*) FROM dbo.DIM_UNIT")).scalar()
        if unit_count == 0:
            print("executing dbo.PROC_POPULATE_DIM_UNIT")
            conn.execute(sa.text("EXEC dbo.PROC_POPULATE_DIM_UNIT;"))
        else:
            print("DIM_UNIT already has data")

    #procedures that load DIM_LOCATION and FACT_WEATHER_FORECAST should execute in every run
        conn.execute(sa.text("EXEC dbo.PROC_POPULATE_DIM_LOCATION;"))
        conn.execute(sa.text("EXEC dbo.PROC_FACT_WEATHER_FORECAST;"))

except sa.exc.DBAPIError as db_err:
    #logging SQL Server errors
    error_msg = f"SQL Server error for Weather datamart ETL: {db_err.orig}"
    print(error_msg)
    logging.error(error_msg)

except Exception as gen_err:
    #logging general Python errors
    error_msg = f"SQL Server error for Weather datamart ETL: {gen_err}"
    print(error_msg)
    logging.error(error_msg)

connected to WeatherDB
DIM_DATE already has data
DIM_UNIT already has data


C:\Users\Matej\anaconda3\Lib\contextlib.py:141: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  return next(self.gen)
